<a href="https://colab.research.google.com/github/Li2angel/BatteryInsight/blob/main/notebooks/01_dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing

## get the data

In [1]:
# cloning the NASA github repo
!git clone --depth 1 https://github.com/XiuzeZhou/NASA.git nasa_repo
DATA_DIR = 'nasa_repo/dataset'   # contains B0005/6/7/18 .mat

Cloning into 'nasa_repo'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 0), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 54.44 MiB | 30.18 MiB/s, done.


## get the loader


preprocessing.py:  

BatteryInsight data loader for the NASA Li-Ion Aging Dataset.

VERIFIED on the real .mat files (B0005, B0006, B0007, B0018).
On B0005 this produces:
    - 168 discharge cycles
    - SOH 1.000 -> 0.714  (initial capacity 1.8565 Ah)
    - Per-sample SOC cleanly in [0, 1]


**DATA SOURCE**

Official: https://data.nasa.gov/dataset/li-ion-battery-aging-datasets  (PCoE).
The four standard cells are B0005, B0006, B0007, B0018, in BatteryAgingARC-FY08Q4.
Put the .mat files in  data/raw/  before running.

**.mat STRUCTURE (so you understand what's happening)**

    mat[<battery_name>][0,0]['cycle'][0]  -> array of cycles
    each cycle has: 'type' ('charge'|'discharge'|'impedance'),
                    'ambient_temperature', 'time', 'data'
    discharge 'data' fields:
        Voltage_measured, Current_measured, Temperature_measured,
        Current_load, Voltage_load, Time (seconds), Capacity (Ah, one value/cycle)


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat

In [3]:
def _cycles(path: str, name: str):
    mat = loadmat(path)
    return mat[name][0, 0]['cycle'][0]

In [4]:
def load_battery(path: str, name: str):
    """Return (samples_df, cycles_df) for one battery.

    samples_df : one row per measurement sample (for SOC modelling, Module 1)
        columns: battery, cycle, voltage, current, temperature, time, soc,
                 capacity, soh, amb_temp
    cycles_df  : one row per discharge cycle (for SOH / degradation, Modules 2 & 3)
        columns: battery, cycle, capacity, soh, amb_temp, mean_temp,
                 max_v, min_v, charge_time(s)
    """
    samples, summary = [], []
    dc = 0
    for c in _cycles(path, name):
        if c['type'][0] != 'discharge':
            continue
        d = c['data'][0, 0]
        cap = float(d['Capacity'][0][0])
        amb = float(c['ambient_temperature'][0][0])
        V = d['Voltage_measured'][0]
        I = d['Current_measured'][0]
        T = d['Temperature_measured'][0]
        t = d['Time'][0]
        dc += 1

        # --- SOC via Coulomb counting within this discharge cycle ---
        # t is in seconds; integrate |I| dt -> Ah drawn; SOC = 1 - drawn/total
        dt = np.diff(t, prepend=t[0])
        ah_drawn = np.cumsum(np.abs(I) * dt) / 3600.0
        total = ah_drawn[-1] if ah_drawn[-1] > 0 else cap
        soc = 1.0 - ah_drawn / total

        for k in range(len(V)):
            samples.append((name, dc, V[k], I[k], T[k], t[k], soc[k], cap, amb))
        summary.append((name, dc, cap, amb, float(np.mean(T)),
                        float(np.max(V)), float(np.min(V)), float(t[-1])))

    sdf = pd.DataFrame(samples, columns=[
        'battery', 'cycle', 'voltage', 'current', 'temperature',
        'time', 'soc', 'capacity', 'amb_temp'])
    cdf = pd.DataFrame(summary, columns=[
        'battery', 'cycle', 'capacity', 'amb_temp', 'mean_temp',
        'max_v', 'min_v', 'charge_time'])

    # SOH = capacity / first measured capacity of THIS battery
    init = cdf['capacity'].iloc[0]
    cdf['soh'] = cdf['capacity'] / init
    sdf = sdf.merge(cdf[['cycle', 'soh']], on='cycle', how='left')
    return sdf, cdf

In [5]:
def load_all(raw_dir: str = 'data/raw',
             batteries=('B0005', 'B0006', 'B0007', 'B0018')):
    """Load several batteries and concatenate. Returns (samples_df, cycles_df)."""
    s_list, c_list = [], []
    for b in batteries:
        p = Path(raw_dir) / f'{b}.mat'
        if not p.exists():
            print(f'[skip] {p} not found')
            continue
        s, c = load_battery(str(p), b)
        s_list.append(s)
        c_list.append(c)
        print(f'[ok] {b}: {len(c)} discharge cycles, '
              f'SOH {c.soh.iloc[0]:.3f} -> {c.soh.iloc[-1]:.3f}')
    return pd.concat(s_list, ignore_index=True), pd.concat(c_list, ignore_index=True)

In [8]:
if __name__ == '__main__':
    # quick smoke test
    s, c = load_all(raw_dir=DATA_DIR)
    print('\nsamples:', s.shape, '| cycles:', c.shape)
    print(c.groupby('battery').agg(cycles=('cycle', 'max'),
                                   soh_end=('soh', 'min')))

[ok] B0005: 168 discharge cycles, SOH 1.000 -> 0.714
[ok] B0006: 168 discharge cycles, SOH 1.000 -> 0.583
[ok] B0007: 168 discharge cycles, SOH 1.000 -> 0.757
[ok] B0018: 132 discharge cycles, SOH 1.000 -> 0.723

samples: (185721, 10) | cycles: (636, 9)
         cycles   soh_end
battery                  
B0005       168  0.693488
B0006       168  0.566893
B0007       168  0.740569
B0018       132  0.722937


## load and sanity-check:

In [7]:
samples_df, cycles_df = load_all(raw_dir=DATA_DIR)
cycles_df.groupby('battery').agg(cycles=('cycle','max'), soh_end=('soh','min'))

[ok] B0005: 168 discharge cycles, SOH 1.000 -> 0.714
[ok] B0006: 168 discharge cycles, SOH 1.000 -> 0.583
[ok] B0007: 168 discharge cycles, SOH 1.000 -> 0.757
[ok] B0018: 132 discharge cycles, SOH 1.000 -> 0.723


,cycles,soh_end
battery,,
B0005,168,0.693488
B0006,168,0.566893
B0007,168,0.740569
B0018,132,0.722937
